## Statistical Analysis of Temperature Data

Second part of Tutorial on Weather Derivatives. 

Quick Summary - our goal:

Aim: we want to price temperature options.

Underlying: HDD/CDD index over given period.

The underlying over a temperature option is the heating/cooling degree days (HDD/CDD) index based on 'approximation' of average temperature and reference (/base) temperature.

References:

[1] Statistical Analysis of Financial Data in R (Rene Carmona, 2014)

### Dataset
Weather Observations for Sydney, Australia - Observatory Hill
Dataset Start to End:

Weather Station 1: 1-Jan 1859 - 30-Aug 2020
Weather Station 2: 18-Oct 2017 - 3-Jul 2022
Data available:

Maximum Temperature
Minimum Temperature

In [3]:
!pip3 install statsmodels --upgrade

In [4]:
import os
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from pathlib import Path

In [5]:
subfolder_path = Path.cwd() / '2_data'
csv_path = subfolder_path / 'max_temp.csv'

max_temp = pd.read_csv(subfolder_path / 'max_temp.csv')
min_temp = pd.read_csv(subfolder_path / 'min_temp.csv')

max_temp.head()

,Product code,Bureau of Meteorology station number,Year,Month,Day,Tmax,Days of accumulation of maximum temperature,Quality
0,IDCJAC0010,66062,1859,1,1,24.4,NaN,Y
1,IDCJAC0010,66062,1859,1,2,24.4,1.0,Y
2,IDCJAC0010,66062,1859,1,3,24.2,1.0,Y
3,IDCJAC0010,66062,1859,1,4,24.7,1.0,Y
4,IDCJAC0010,66062,1859,1,5,24.6,1.0,Y


### Check for mission data

In [6]:
max_temp.isnull().value_counts(),min_temp.isna().value_counts()

(Product code  Bureau of Meteorology station number  Year   Month  Day    Tmax   Days of accumulation of maximum temperature  Quality
 False         False                                 False  False  False  False  False                                        False      59428
                                                                          True   True                                         True         151
                                                                          False  True                                         False        135
                                                                                 False                                        True           2
                                                                          True   False                                        True           2
 Name: count, dtype: int64,
 Product code  Bureau of Meteorology station number  Year   Month  Day    Tmin   Days of accumulation of minimum temperatur

In [7]:
count = 0
for mx, mn in zip(np.where(max_temp.isnull())[0], np.where(min_temp.isnull())[0]):
    if mx != mn:
        count += 1

print('Number of Misaligned Null Values: ', count)

Number of Misaligned Null Values:  41


### Clean min/max data

In [8]:
def datetime(row):
    return dt.datetime(row.Year,row.Month,row.Day)

In [9]:
max_temp['Date'] = max_temp.apply(datetime,axis=1)
min_temp['Date'] = min_temp.apply(datetime,axis=1)
max_temp.set_index('Date', inplace=True)
min_temp.set_index('Date', inplace=True)
drop_cols = [0,1,2,3,4,6,7]
max_temp.drop(max_temp.columns[drop_cols],axis=1,inplace=True)
min_temp.drop(min_temp.columns[drop_cols],axis=1,inplace=True)
max_temp.rename(columns={'Maximum temperature (Degree C)':'Tmax'}, inplace=True)
min_temp.rename(columns={'Minimum temperature (Degree C)':'Tmin'}, inplace=True)

In [10]:
temps = max_temp.merge(min_temp,how='inner',left_on=['Date'],right_on=['Date'])

def avg_temp(row):
    return (row.Tmax+row.Tmin)/2

temps['T'] = temps.apply(avg_temp,axis=1)

# drop na values here
temps = temps.dropna()
temps

,Tmax,Tmin,T
Date,,,
1859-01-01,24.4,14.5,19.45
1859-01-02,24.4,15.7,20.05
1859-01-03,24.2,15.3,19.75
1859-01-04,24.7,17.4,21.05
1859-01-05,24.6,16.9,20.75
...,...,...,...
2022-06-29,17.9,8.0,12.95
2022-06-30,16.9,10.2,13.55
2022-07-01,12.0,10.3,11.15


In [11]:
temps_season = temps.copy(deep=True)
temps_season['month'] = temps_season.index.month
mask = (temps_season['month'] >= 5) & (temps_season['month'] <= 10)
temps_season['winter'] = np.where(mask,1,0)
temps_season['summer'] = np.where(temps_season['winter'] != 1,1,0)
temps_season

,Tmax,Tmin,T,month,winter,summer
Date,,,,,,
1859-01-01,24.4,14.5,19.45,1,0,1
1859-01-02,24.4,15.7,20.05,1,0,1
1859-01-03,24.2,15.3,19.75,1,0,1
1859-01-04,24.7,17.4,21.05,1,0,1
1859-01-05,24.6,16.9,20.75,1,0,1
...,...,...,...,...,...,...
2022-06-29,17.9,8.0,12.95,6,1,0
2022-06-30,16.9,10.2,13.55,6,1,0
2022-07-01,12.0,10.3,11.15,7,1,0
